# 05 — Roofline atlas

**Question.** Where do the measured compute and bandwidth ceilings meet for each precision and memory configuration, and where do achieved kernels sit relative to them?

Line style tells the origin of a ceiling: **dashed** = theoretical (vendor or derived), **solid** = best measured by a dedicated kernel, **dotted** = what a library kernel (oneDNN GEMM) achieved. Each memory ceiling has one purpose. Sections 1 and 3 draw the **ERT reference envelope**: the best sustained rate of every kernel (read-only, Triad, non-temporal Triad, copy), which on Flat HBM is copy. Every LLM statement uses the **read-only** ceiling (weights are only read): the GEMM figures here, the headline roofline, Table 2 and the decode bound of the master notebook. The FP64 sweep uses the **copy** ceiling (it loads one array and stores another), and Cache mode is read at a stated working set. Bytes are modeled, so these are empirical envelopes, not counter-based bounds. Section 6 lists every point that lies above its own roofline.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench import report, roofline, plotting
from hwbench.notebook import setup
ctx = setup('05_roofline_atlas')
thesis, ext, theory = ctx.thesis, ctx.ext, ctx.theory
table, figure, say, say_claims = ctx.table, ctx.figure, ctx.say, ctx.say_claims
from hwbench import claims

In [ ]:
PATHS = ['fp64', 'fp32', 'avx512_bf16', 'vnni_int8', 'amx_bf16', 'amx_int8']
def compute_set(mode, scope, cores):
    out = [report.compute_ceiling(ext, p, mode, cores, scope) for p in PATHS]
    return out
def theory_set(paths, scope, cores):
    return [report.theoretical_ceiling(theory, p, cores, scope) for p in paths]

## 1. Flat mode, one socket (56 cores)

In [ ]:
mem = [report.memory_ceiling(ext, thesis, 'flat', 'hbm'), report.memory_ceiling(ext, thesis, 'flat', 'ddr')]
fig = plotting.roofline_figure(theory_set(['fp64', 'amx_bf16', 'amx_int8'], '1S', 56) + compute_set('flat', '1S', 56), mem,
                               title='Flat · one socket · ERT envelope (best kernel) and vendor peaks (dashed)')
figure(fig, 'atlas_flat_1s')
table(pd.DataFrame([dict(memory=f'{m.tier} ({m.kernel})', GBs=m.gbs, **{p: roofline.ridge(c.gops, m.gbs) for p, c in zip(PATHS, compute_set('flat', '1S', 56))}) for m in mem]))

The table gives the **ridge** (operations per byte) of the ERT envelope, at which each path stops being memory-bound. Higher-throughput paths need proportionally more reuse. The ridges quoted for LLM work (master Table 2, below) use the read-only ceiling instead.

In [ ]:
say_claims(claims.ridges(ext, thesis, theory), ['ridge_points'])

## 2. Cache mode, one socket

In [ ]:
resident = report.memory_ceiling(ext, thesis, 'cache', 'effective_cache', access='read', at_gib=16)
spilled = report.memory_ceiling(ext, thesis, 'cache', 'effective_cache', at_gib=96)
fig = plotting.roofline_figure(theory_set(['amx_bf16', 'amx_int8'], '1S', 56) + compute_set('cache', '1S', 56), [resident, spilled],
                               title='Cache · one socket · data resident in HBM (16 GiB) and spilled to DDR (96 GiB)')
figure(fig, 'atlas_cache_1s')
table(pd.DataFrame([dict(case=c.tier, kernel=c.kernel, GBs=c.gbs, working_set_gib=c.working_set_gib) for c in (resident, spilled)]))

In [ ]:
say_claims(claims.cache_mode(ext, thesis), ['cache_declines_before_capacity', 'cache_triad_spilled_ceiling', 'cache_spilled_gemm'])

Every ridge moves right by the ratio of the two slopes. The 16 GiB line is the case of a model of that size; the 96 GiB line is the Triad-based ceiling of the spilled case. Spilled, the kernels disagree by almost a factor of two and a GEMM streaming weights is faster than the read-only kernel (below): read-only is therefore not a ceiling for a spilled GEMM, and Triad, the fastest kernel measured there, is used.

## 3. Flat mode, both sockets

In [ ]:
mem2 = [report.memory_ceiling(ext, thesis, 'flat', 'hbm', '2S'), report.memory_ceiling(ext, thesis, 'flat', 'ddr', '2S')]
fig = plotting.roofline_figure(theory_set(['fp64', 'amx_bf16', 'amx_int8'], '2S', 56) + compute_set('flat', '2S', 56), mem2,
                               title='Flat · both sockets (two timed processes) · ERT envelope and vendor ceilings')
figure(fig, 'atlas_flat_2s')

In [ ]:
say_claims(claims.ridges(ext, thesis, theory), ['ridge_points', 'two_sockets_ridge'])

## 4. Achieved points: FP64 sweep

In [ ]:
flat_hbm, flat_ddr = mem
fp64 = [report.compute_ceiling(ext, 'fp64', 'flat', 56)]
mem_rw = [report.sweep_ceiling(ext, thesis, 'hbm'), report.sweep_ceiling(ext, thesis, 'ddr')]
pts = report.fp64_sweep_points(thesis, 'flat', 'hbm') + report.fp64_sweep_points(thesis, 'flat', 'ddr')
pts12 = report.fp64_sweep12_points(ext, thesis, theory, 'hbm') + report.fp64_sweep12_points(ext, thesis, theory, 'ddr')
audits = [roofline.audit_points(pts, mem_rw, fp64).assign(group='FP64 sweep · Flat · sweep ceiling'),
          roofline.audit_points(pts12, mem_rw, fp64).assign(group='FP64 sweep, 12 chains · Flat · sweep ceiling')]
fig = plotting.roofline_figure(fp64 + theory_set(['fp64'], '1S', 56), mem_rw, pts + pts12, title='FP64 intensity sweep · 8 chains (October) and 12 chains · Flat · one socket', x_range=(0.05, 256))
figure(fig, 'atlas_fp64_sweep')
hb = [p for p in pts if p.memory_target == 'hbm']
say_claims(claims.fp64_sweep(ext, thesis, theory))

## 5. Achieved points: BF16 and INT8 GEMM

Each point is one (shape, M) pair of the first sweep. Intensity rises with M because the same weights are reused for more tokens; the point leaves the memory slope when the GEMM becomes compute-limited. The first sweep's pool was about twice the on-chip capacity, so its points at M ≤ 32 are drawn **hollow** (weight traffic overstated); the converged sweep of 14336×4096 (16× pool, M = 1 to 256) and the 64× pool points of 4096×4096 (M = 1, 16, 256) are drawn as their own series.

In [ ]:
gm = report.gemm_points(ext)
mem_read = [report.memory_ceiling(ext, thesis, 'flat', 'hbm', access='read'), report.memory_ceiling(ext, thesis, 'flat', 'ddr', access='read')]
for dtype, path in [('bf16', 'amx_bf16'), ('int8', 'amx_int8')]:
    sel = gm[(gm.memory_mode == 'flat') & (gm.dtype == dtype)]
    pts = report.gemm_workload_points(sel, theory)
    sweep = report.converged_sweep(ext)
    conv = report.gemm_workload_points(sweep[(sweep.dtype == dtype) & ((sweep.layout == 'plain') | (sweep.dtype == 'int8'))], theory)
    for q in conv:
        q.provenance['series'] = '14336x4096 · converged sweep (16× pool)'
    conv += [p for p in report.gemm_pool_points(ext, 64, '4096x4096') if p.precision == dtype]
    audits.append(roofline.audit_points(conv, mem_read, [report.compute_ceiling(ext, path, 'flat', 56)]).assign(group=f'{dtype.upper()} GEMM · Flat · converged pools · read-only ceiling'))
    cs = [report.compute_ceiling(ext, path, 'flat', 56), report.theoretical_ceiling(theory, path, 56)]
    if dtype == 'bf16':
        cs.append(report.compute_ceiling(ext, 'avx512_bf16', 'flat', 56))
    else:
        cs.append(report.compute_ceiling(ext, 'vnni_int8', 'flat', 56))
    audits.append(roofline.audit_points(pts, mem_read, cs).assign(group=f'{dtype.upper()} GEMM · Flat · read-only ceiling'))
    fig = plotting.roofline_figure(cs, mem_read, pts + conv, title=f'{dtype.upper()} GEMM M-sweep at Llama-3.1-8B shapes · Flat · one socket', height=820)
    figure(fig, f'atlas_gemm_{dtype}')

In [ ]:
rows = []
for dtype, path in [('bf16', 'amx_bf16'), ('int8', 'amx_int8')]:
    peak = report.compute_ceiling(ext, path, 'flat', 56).gops
    theo = theory.peak_gops(path, 56)
    for tgt, bw in [('hbm', flat_hbm.gbs), ('ddr', flat_ddr.gbs)]:
        sel = gm[(gm.memory_mode == 'flat') & (gm.dtype == dtype) & (gm.memory_target == tgt)]
        best = sel.loc[sel.gops.idxmax()]
        rows.append(dict(dtype=dtype, memory=tgt, best_shape=best['shape'], best_M=int(best.m), achieved_TOPs=best.gops / 1000,
                         measured_tile_peak_TOPs=peak / 1000, theory_TOPs=theo / 1000,
                         achieved_over_measured=best.gops / peak, measured_over_theory=peak / theo, achieved_over_theory=best.gops / theo))
gap = pd.DataFrame(rows)
table(gap)

### Counted against modeled intensity

The cores' DRAM counters give the intensity a point actually has. Each BF16 and INT8 point of the 14336×4096 projection appears twice, with modeled and with counted bytes, at the same rate.

In [ ]:
cs = [report.compute_ceiling(ext, p, 'flat', 56) for p in ('amx_bf16', 'amx_int8')]
fig = plotting.roofline_figure(cs, [mem_read[0]], report.counted_gemm_points(ext, theory, 'hbm'), title='GEMM 14336×4096 · Flat HBM · modeled and counted bytes', point_labels=True, x_range=(0.5, 4000))
figure(fig, 'atlas_counted_gemm')
say_claims(claims.counted(ext, thesis), ['counted_gemm_weights', 'counted_gemm_prefill'])

**Gap analysis.** Two layers separate the vendor number from library performance: (1) the vendor peak versus what a hand-written tile kernel sustains (the clock the cores actually ran at; compare `measured_over_theory` below); (2) the tile-kernel peak versus an optimized library GEMM (the achieved-over-measured column).

## 6. Ceiling audit: points above their own roofline

A ceiling is only an upper bound if no accepted point lies above it. Each point below is compared with `min(compute ceiling, bandwidth ceiling × intensity)` for its own tier and ISA path; it counts as *exceeding* when it is above that bound by more than the larger of its own and the ceiling's run-to-run spread (CV). Flat GEMM uses the read-only ceiling (weights are only read). Cache-mode GEMM uses the best-kernel ceiling at the pool size of the run (16 GiB resident, 96 GiB spilled), because spilled GEMMs stream faster than the read-only kernel (section 2), so read-only would not bound them.

In [ ]:
for pool, ceiling in [(16, resident), (96, spilled)]:
    for dtype, path in [('bf16', 'amx_bf16'), ('int8', 'amx_int8')]:
        sel = gm[(gm.memory_mode == 'cache') & (gm.dtype == dtype) & ((gm.pool_gib - pool).abs() < 0.5)]
        audits.append(roofline.audit_points(report.gemm_workload_points(sel, theory), [ceiling], compute_set('cache', '1S', 56)).assign(group=f'{dtype.upper()} GEMM · Cache · {pool} GiB pool · best-kernel ceiling'))
audits.append(roofline.audit_points(report.gemm_pool_points(ext, 64), mem_read, [report.compute_ceiling(ext, p, 'flat', 56) for p in ('amx_bf16', 'amx_int8')]).assign(group='GEMM · Flat · 64× pool · read-only ceiling'))
audit = pd.concat(audits, ignore_index=True)
summary = audit.groupby('group').agg(points=('exceeds', 'size'), exceeding=('exceeds', 'sum'), max_ratio=('ratio', 'max')).reset_index()
table(summary)
over = audit[audit.exceeds]
say(f"**Result.** {len(over)} of {len(audit)} points lie above their roofline by more than their spread; the largest excess is {audit.ratio.max() - 1:.1%}. "
    f"{int(over.unconverged.sum())} of them are hollow first-sweep points (M ≤ 32, pool too small); {int((~over.unconverged).sum())} are filled.")
assert over.unconverged.all(), 'a converged or large-M point lies above its ceiling'
table(over.sort_values('ratio', ascending=False)[['group', 'series', 'intensity', 'bandwidth_gbs', 'ceiling_gbs', 'ceiling_kernel', 'ratio', 'tolerance_percent', 'unconverged']])

**Reading the exceptions.** They are listed, not hidden. Two mechanisms can raise a point above a kernel-based ceiling: the access pattern differs from every ceiling kernel, or the modeled bytes overcount real traffic. The counters and the pool sweep tell them apart:

In [ ]:
g64 = audit[audit.group.str.contains('64× pool')]
say(f"**Result.** With the converged 64× pool {int(g64.exceeds.sum())} of {len(g64)} GEMM points lie above their roofline.")
say_claims(claims.gemm(ext, thesis, theory, ROOT), ['gemm_pool_convergence'])
say_claims(claims.counted(ext, thesis), ['counted_gemm_weights', 'sweep_exceeds_triad_ceiling'])

The DDR GEMM exceptions of the first sweep were cache reuse in a pool that was too small (counted traffic below modeled; none remain with the 64× pool). The HBM sweep exceedance against Triad was a property of the ceiling, not of the point: a one-load-one-store stream sustains more than Triad, and with the copy kernel as its ceiling the sweeps lie within it. What is listed above is therefore only the first sweep's small-pool GEMM points on DDR, kept for provenance.